# Colab preflight (optional diagnostic)

Use this only to diagnose GPU, Drive, clone, or package setup. It is **not** a prerequisite for the canonical reproduction notebook, which includes its own setup: [`01_reproduce_mft_gemma3.ipynb`](https://colab.research.google.com/github/rlogger/em-displacement-vlm/blob/main/notebooks/01_reproduce_mft_gemma3.ipynb).

Runtime → GPU → **A100** recommended.

In [1]:
!nvidia-smi
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

Thu Jul 30 18:34:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P0             52W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Drive mount

In [2]:
from pathlib import Path
import os

MOUNT_DRIVE = True
DRIVE_PROJECT = Path("/content/drive/MyDrive/em-displacement-vlm")

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    for sub in ("data", "checkpoints", "results"):
        (DRIVE_PROJECT / sub).mkdir(parents=True, exist_ok=True)
    os.environ["EM_DATA_DIR"] = str(DRIVE_PROJECT / "data")
    os.environ["EM_CHECKPOINT_DIR"] = str(DRIVE_PROJECT / "checkpoints")
    os.environ["EM_RESULTS_DIR"] = str(DRIVE_PROJECT / "results")
    print("Drive project:", DRIVE_PROJECT)
else:
    print("Drive mount skipped — using /content for ephemeral storage.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive project: /content/drive/MyDrive/em-displacement-vlm


## 2. Clone / pull

In [3]:
from pathlib import Path

REPO_URL = "https://github.com/rlogger/em-displacement-vlm.git"
REPO_DIR = Path("/content/em-displacement-vlm")
BRANCH = "main"

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git rev-parse --short HEAD
!git status -sb

/content/em-displacement-vlm
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/rlogger/em-displacement-vlm
 * branch            main       -> FETCH_HEAD
Already up to date.
c013423
## main...origin/main


## 3. Install

In [4]:
import sys
sys.path.insert(0, str(REPO_DIR))

%pip install -q -e ".[vlm,dev]"

from em_displacement_vlm.runtime import runtime_info
from em_displacement_vlm.paths import data_dir, checkpoint_dir

for k, v in runtime_info().items():
    print(f"{k}: {v}")
print("data_dir:", data_dir())
print("checkpoint_dir:", checkpoint_dir())
print("\nFor the canonical M_ft reproduction open: notebooks/01_reproduce_mft_gemma3.ipynb")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for em-displacement-vlm (pyproject.toml) ... done
python: 3.12.13
platform: Linux-6.6.122+-x86_64-with-glibc2.35
colab: True
repo_root: /content/em-displacement-vlm
packages: {'accelerate': '1.14.0', 'datasets': '4.0.0', 'huggingface-hub': '1.23.0', 'numpy': '2.0.2', 'peft': '0.19.1', 'safetensors': '0.8.0', 'transformers': '5.13.1', 'trl': 'not installed', 'unsloth': 'not installed', 'unsloth-zoo': 'not installed', 'wandb': '0.28.0'}
torch: 2.11.0+cu128
cuda_runtime: 12.8
cudnn: 91900
cuda_available: True
cuda_device: NVIDIA A100-SXM4-40GB
data_dir: /content/drive/MyDrive/em-displacement-vlm/data
checkpoint_dir: /content/drive/MyDrive/em-displacement-vlm/checkpoints

For the canonical M_ft reproduction open: notebooks/

## 4. Secrets

In [5]:
from google.colab import userdata
import os

def _set_secret(name: str) -> None:
    try:
        os.environ[name] = userdata.get(name)
        print(f"Loaded secret: {name}")
    except Exception:
        print(f"Secret not set (ok if unused): {name}")

for key in ("HF_TOKEN",):
    _set_secret(key)

if os.environ.get("HF_TOKEN"):
    !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential

Loaded secret: HF_TOKEN

Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help

